# Paper #4: Perceptrons (Minsky & Papert, 1969)

## Implementation: Proving and Visualizing the Limitations of Single-Layer Perceptrons

This notebook demonstrates the key results from Minsky & Papert's book:

1. **XOR Impossibility**: Proving and visualizing why XOR is not linearly separable
2. **Parity Problem**: Generalizing XOR to $n$ bits — impossible for any fixed-order perceptron
3. **Connectedness**: Why local predicates cannot detect global structure
4. **Breaking Through**: How a 2-layer network solves XOR (preview of Paper #6)
5. **Universal Approximation**: Glimpse of what multi-layer networks can do

**Prerequisites**: Paper #3 (Perceptron learning rule, linear separability)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import FancyArrowPatch
from itertools import product

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## Part 1: XOR — The Simplest Impossible Problem

XOR is the smallest Boolean function that a single-layer perceptron cannot compute.

**Minsky & Papert's proof** shows that no values of $\alpha_1$, $\alpha_2$, $\theta$ satisfy
all four XOR constraints simultaneously. Let's visualize why.

In [ ]:
# XOR truth table
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: XOR is not linearly separable ---
ax = axes[0]
for i, (x, y) in enumerate(zip(X_xor, y_xor)):
    color = 'blue' if y == 1 else 'red'
    marker = 'o' if y == 1 else 's'
    ax.scatter(*x, c=color, s=300, edgecolors='black', linewidth=2,
              marker=marker, zorder=5)
    ax.annotate(f'({x[0]},{x[1]})→{y}', xy=x, xytext=(10, 10),
               textcoords='offset points', fontsize=11)

# Try several lines — none work
x_line = np.linspace(-0.3, 1.3, 100)
for slope, intercept, ls in [(-1, 0.8, '--'), (-1, 0.3, '-.'), (-0.5, 0.7, ':')]:
    ax.plot(x_line, slope * x_line + intercept, ls, color='gray', alpha=0.5, linewidth=1.5)

ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('XOR: No line separates the classes\n(Red=0, Blue=1)')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# --- Plot 2: AND is linearly separable ---
y_and = np.array([0, 0, 0, 1])
ax = axes[1]
for i, (x, y) in enumerate(zip(X_xor, y_and)):
    color = 'blue' if y == 1 else 'red'
    marker = 'o' if y == 1 else 's'
    ax.scatter(*x, c=color, s=300, edgecolors='black', linewidth=2,
              marker=marker, zorder=5)

# Decision boundary for AND: x1 + x2 = 1.5
ax.plot(x_line, 1.5 - x_line, 'k-', linewidth=2)
ax.fill_between(x_line, 1.5 - x_line, 1.5, alpha=0.1, color='blue')
ax.fill_between(x_line, -0.5, 1.5 - x_line, alpha=0.1, color='red')
ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('AND: Linearly separable ✓')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# --- Plot 3: OR is linearly separable ---
y_or = np.array([0, 1, 1, 1])
ax = axes[2]
for i, (x, y) in enumerate(zip(X_xor, y_or)):
    color = 'blue' if y == 1 else 'red'
    marker = 'o' if y == 1 else 's'
    ax.scatter(*x, c=color, s=300, edgecolors='black', linewidth=2,
              marker=marker, zorder=5)

# Decision boundary for OR: x1 + x2 = 0.5
ax.plot(x_line, 0.5 - x_line, 'k-', linewidth=2)
ax.fill_between(x_line, 0.5 - x_line, 1.5, alpha=0.1, color='blue')
ax.fill_between(x_line, -0.5, 0.5 - x_line, alpha=0.1, color='red')
ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('OR: Linearly separable ✓')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.suptitle('Minsky & Papert\'s Key Insight: XOR is NOT Linearly Separable', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Algebraic Proof: XOR Cannot Be Computed by a Linear Threshold Function

Let's verify the impossibility by exhaustive search over all possible weight combinations.

In [ ]:
def linear_threshold(x: np.ndarray, w: np.ndarray, theta: float) -> int:
    """Compute a linear threshold function: 1 if w·x > theta, else 0."""
    return 1 if np.dot(w, x) > theta else 0


def check_separability(X: np.ndarray, y: np.ndarray, resolution: int = 50) -> dict:
    """Exhaustively search for weights that linearly separate the data.
    
    Searches over a grid of (w1, w2, theta) values.
    Returns the best solution found, or None if no perfect solution exists.
    """
    best = {'w1': None, 'w2': None, 'theta': None, 'correct': 0}
    search_range = np.linspace(-3, 3, resolution)
    
    for w1 in search_range:
        for w2 in search_range:
            for theta in search_range:
                w = np.array([w1, w2])
                predictions = [linear_threshold(x, w, theta) for x in X]
                correct = sum(p == t for p, t in zip(predictions, y))
                if correct > best['correct']:
                    best = {'w1': w1, 'w2': w2, 'theta': theta, 'correct': correct}
                if correct == len(y):
                    return best
    
    return best


# Test all four 2-input Boolean functions
functions = {
    'AND':  np.array([0, 0, 0, 1]),
    'OR':   np.array([0, 1, 1, 1]),
    'NAND': np.array([1, 1, 1, 0]),
    'XOR':  np.array([0, 1, 1, 0]),
    'XNOR': np.array([1, 0, 0, 1]),
}

print("Exhaustive search for linear separability")
print("=" * 55)
for name, y in functions.items():
    result = check_separability(X_xor, y, resolution=30)
    separable = result['correct'] == 4
    status = '✓ Separable' if separable else '✗ NOT separable'
    print(f"  {name:5s}: {status} (best: {result['correct']}/4 correct)")
    if separable:
        print(f"         w=({result['w1']:.2f}, {result['w2']:.2f}), θ={result['theta']:.2f}")

print(f"\n→ XOR and XNOR are the ONLY non-linearly-separable 2-input functions.")
print(f"  This is exactly what Minsky & Papert proved.")

## Part 2: Parity — XOR Generalized to $n$ Bits

Parity returns 1 if an odd number of inputs are 1, and 0 if even.
For $n=2$, parity = XOR. Minsky & Papert proved that parity of $n$ bits
requires a perceptron of order $n$ — every predicate must see ALL inputs.

Let's demonstrate: as $n$ grows, a single-layer perceptron's failure becomes dramatic.

In [ ]:
def parity(x: np.ndarray) -> int:
    """Compute parity: 1 if odd number of 1s, 0 if even."""
    return int(np.sum(x) % 2)


def train_perceptron(X, y, n_epochs=200, lr=0.1):
    """Train a single-layer perceptron. Returns final accuracy."""
    n_features = X.shape[1]
    w = np.random.randn(n_features) * 0.1
    b = 0.0
    
    for epoch in range(n_epochs):
        for xi, yi in zip(X, y):
            y_target = 2 * yi - 1  # Convert 0/1 to -1/+1
            activation = np.dot(w, xi) + b
            pred = 1 if activation >= 0 else -1
            if pred != y_target:
                w += lr * y_target * xi
                b += lr * y_target
    
    predictions = [1 if np.dot(w, x) + b >= 0 else 0 for x in X]
    accuracy = np.mean(np.array(predictions) == y)
    return accuracy


# Test parity for increasing n
print("Parity Problem: Single-Layer Perceptron Performance")
print("=" * 55)

n_values = [2, 3, 4, 5, 6, 7, 8]
accuracies = []
max_possible = []

for n in n_values:
    # Generate all 2^n input patterns
    X = np.array(list(product([0, 1], repeat=n)))
    y = np.array([parity(x) for x in X])
    
    # Try multiple random initializations
    best_acc = max(train_perceptron(X, y) for _ in range(10))
    accuracies.append(best_acc)
    
    # Maximum achievable by chance: always predict majority class
    majority = max(np.mean(y), 1 - np.mean(y))
    max_possible.append(majority)
    
    print(f"  n={n}: {2**n:4d} patterns, best accuracy = {best_acc:.1%} "
          f"(chance = {majority:.1%})")

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(range(len(n_values)), accuracies, color='steelblue', alpha=0.8, label='Perceptron accuracy')
ax.plot(range(len(n_values)), max_possible, 'r--o', label='Chance level', linewidth=2)
ax.set_xticks(range(len(n_values)))
ax.set_xticklabels([f'n={n}' for n in n_values])
ax.set_ylabel('Accuracy')
ax.set_title('Parity Problem: Single-Layer Perceptron Cannot Do Better Than Chance\n'
             '(Minsky & Papert proved this mathematically)')
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n→ The perceptron never exceeds chance level on parity!")
print("  Parity requires order-n: every predicate must see ALL inputs.")
print("  No partial observation can determine parity.")

## Part 3: Connectedness — Beyond Any Finite Order

Connectedness detection requires global structure analysis.
A predicate with a limited "field of view" cannot trace paths across the retina.

Let's demonstrate with a simple grid world.

In [ ]:
def is_connected(grid: np.ndarray) -> bool:
    """Check if active cells in a grid form a single connected component.
    
    Uses flood fill (BFS) from the first active cell.
    """
    rows, cols = grid.shape
    active = set(zip(*np.where(grid == 1)))
    
    if len(active) == 0:
        return True
    
    # BFS from first active cell
    start = next(iter(active))
    visited = {start}
    queue = [start]
    
    while queue:
        r, c = queue.pop(0)
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if (nr, nc) in active and (nr, nc) not in visited:
                visited.add((nr, nc))
                queue.append((nr, nc))
    
    return len(visited) == len(active)


def local_predicates(grid: np.ndarray, window_size: int) -> np.ndarray:
    """Extract features using local predicates of limited order.
    
    Each predicate observes a window_size x window_size region.
    Returns a feature vector of local pattern statistics.
    """
    rows, cols = grid.shape
    features = []
    
    for r in range(rows - window_size + 1):
        for c in range(cols - window_size + 1):
            window = grid[r:r+window_size, c:c+window_size]
            # Features: sum, has_gap, edge_count
            features.append(np.sum(window))
            # Count adjacent active pairs (local connectivity)
            h_pairs = np.sum(window[:, :-1] * window[:, 1:])  # horizontal
            v_pairs = np.sum(window[:-1, :] * window[1:, :])  # vertical
            features.append(h_pairs + v_pairs)
    
    return np.array(features)


# Generate examples: connected vs disconnected patterns
grid_size = 8

# Example patterns
connected_examples = [
    # L-shape
    np.array([[1,0,0,0,0,0,0,0],
              [1,0,0,0,0,0,0,0],
              [1,0,0,0,0,0,0,0],
              [1,1,1,1,0,0,0,0],
              [0,0,0,0,0,0,0,0],
              [0,0,0,0,0,0,0,0],
              [0,0,0,0,0,0,0,0],
              [0,0,0,0,0,0,0,0]]),
    # Winding path
    np.array([[1,1,1,0,0,0,0,0],
              [0,0,1,0,0,0,0,0],
              [0,0,1,1,1,0,0,0],
              [0,0,0,0,1,0,0,0],
              [0,0,0,0,1,1,1,0],
              [0,0,0,0,0,0,1,0],
              [0,0,0,0,0,0,1,1],
              [0,0,0,0,0,0,0,0]]),
]

disconnected_examples = [
    # Two separate blocks
    np.array([[1,1,0,0,0,0,0,0],
              [1,1,0,0,0,0,0,0],
              [0,0,0,0,0,0,0,0],
              [0,0,0,0,0,0,0,0],
              [0,0,0,0,0,0,0,0],
              [0,0,0,0,0,0,0,0],
              [0,0,0,0,0,0,1,1],
              [0,0,0,0,0,0,1,1]]),
    # Almost connected (1-pixel gap)
    np.array([[1,1,1,0,0,0,0,0],
              [0,0,1,0,0,0,0,0],
              [0,0,1,1,0,0,0,0],
              [0,0,0,0,0,0,0,0],
              [0,0,0,0,0,1,1,0],
              [0,0,0,0,0,0,1,0],
              [0,0,0,0,0,0,1,1],
              [0,0,0,0,0,0,0,0]]),
]

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
all_examples = connected_examples + disconnected_examples
titles = ['Connected (L-shape)', 'Connected (winding path)',
          'Disconnected (two blocks)', 'Disconnected (gap)']

for ax, grid, title in zip(axes, all_examples, titles):
    ax.imshow(grid, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    # Add grid lines
    for i in range(grid_size + 1):
        ax.axhline(i - 0.5, color='gray', linewidth=0.5)
        ax.axvline(i - 0.5, color='gray', linewidth=0.5)

plt.suptitle('Connectedness: Can a perceptron tell these apart?', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate that local predicates fail at connectedness
# Generate many random patterns, extract local features, try to classify

def generate_random_patterns(grid_size: int, n_patterns: int, n_active: int):
    """Generate random grid patterns, labeled by connectedness."""
    patterns = []
    labels = []
    
    for _ in range(n_patterns):
        grid = np.zeros((grid_size, grid_size))
        positions = np.random.choice(grid_size * grid_size, size=n_active, replace=False)
        for pos in positions:
            grid[pos // grid_size, pos % grid_size] = 1
        patterns.append(grid)
        labels.append(int(is_connected(grid)))
    
    return patterns, np.array(labels)


# Generate training data
patterns, labels = generate_random_patterns(grid_size=6, n_patterns=500, n_active=8)

# Try classifying with local predicates of increasing window size
window_sizes = [2, 3, 4, 5]

print("Connectedness Classification with Local Predicates")
print("=" * 55)
print(f"Dataset: {len(patterns)} patterns, {sum(labels)} connected, {len(labels)-sum(labels)} disconnected")
print()

local_accuracies = []

for ws in window_sizes:
    # Extract local features
    X = np.array([local_predicates(p, ws) for p in patterns])
    y = 2 * labels - 1  # Convert to -1/+1
    
    # Train a linear classifier (perceptron)
    # Simple: use pseudo-inverse for linear regression as a proxy
    X_bias = np.column_stack([X, np.ones(len(X))])
    try:
        w = np.linalg.lstsq(X_bias, y, rcond=None)[0]
        predictions = np.sign(X_bias @ w)
        accuracy = np.mean(predictions == y)
    except np.linalg.LinAlgError:
        accuracy = 0.5
    
    local_accuracies.append(accuracy)
    print(f"  Window {ws}x{ws} (order-{ws**2:2d}): {X.shape[1]:3d} features, accuracy = {accuracy:.1%}")

# The "cheat" solution: use flood fill (global algorithm)
global_accuracy = 1.0  # BFS always gets it right

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(range(len(window_sizes)), local_accuracies, color='steelblue', alpha=0.8,
              label='Local predicates (perceptron)')
ax.axhline(y=global_accuracy, color='green', linestyle='-', linewidth=2,
           label='Global algorithm (flood fill) = 100%')
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5,
           label='Chance level')

ax.set_xticks(range(len(window_sizes)))
ax.set_xticklabels([f'{ws}×{ws}\n(order {ws**2})' for ws in window_sizes])
ax.set_xlabel('Local predicate window size')
ax.set_ylabel('Classification accuracy')
ax.set_title('Connectedness: Local Predicates vs Global Algorithm\n'
             'Minsky & Papert proved: no finite window size is sufficient')
ax.legend(loc='lower right')
ax.set_ylim(0.4, 1.05)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n→ Local predicates improve with larger windows, but NEVER reach 100%.")
print("  Connectedness fundamentally requires tracing paths of arbitrary length.")
print("  No fixed-size window can capture this — Minsky & Papert's key theorem.")

## Part 4: Breaking Through — Two Layers Solve XOR

Minsky & Papert's limitations apply to **single-layer** perceptrons.
With just **one hidden layer**, XOR becomes trivially solvable.

The key insight: XOR = (x₁ AND NOT x₂) OR (NOT x₁ AND x₂)

Or equivalently: XOR = (x₁ OR x₂) AND NOT (x₁ AND x₂)

Each component (OR, AND, NOT) is linearly separable — so we can compute
them in one layer, then combine in a second layer!

In [ ]:
class TwoLayerXOR:
    """A hand-designed two-layer network that computes XOR.
    
    Architecture:
        Input (x1, x2) → Hidden (h1=OR, h2=NAND) → Output (AND)
    
    XOR(x1, x2) = AND(OR(x1, x2), NAND(x1, x2))
    """
    
    def __init__(self):
        # Hidden layer weights (2 neurons)
        # h1 = OR: w=[1, 1], b=-0.5 → fires if x1+x2 > 0.5
        # h2 = NAND: w=[-1, -1], b=1.5 → fires if -x1-x2 > -1.5
        self.W_hidden = np.array([[1, 1], [-1, -1]], dtype=float)  # (2, 2)
        self.b_hidden = np.array([-0.5, 1.5])  # (2,)
        
        # Output layer weights (1 neuron)
        # AND: w=[1, 1], b=-1.5 → fires if h1+h2 > 1.5
        self.W_output = np.array([1, 1], dtype=float)  # (2,)
        self.b_output = -1.5
    
    def forward(self, x: np.ndarray) -> tuple:
        """Forward pass. Returns (hidden activations, output)."""
        # Hidden layer: step activation
        h = (self.W_hidden @ x + self.b_hidden > 0).astype(float)
        # Output layer: step activation
        out = float(np.dot(self.W_output, h) + self.b_output > 0)
        return h, out


# Verify
net = TwoLayerXOR()

print("Two-Layer Network Solving XOR")
print("=" * 50)
print("Architecture: Input → [OR, NAND] → AND → Output")
print()
print("  x1  x2  │  OR  NAND  │  AND(OR,NAND) = XOR")
print("─" * 50)

for x in X_xor:
    h, out = net.forward(x)
    expected = x[0] ^ x[1]  # Python XOR
    print(f"   {int(x[0])}   {int(x[1])}  │   {int(h[0])}    {int(h[1])}   │       {int(out)}       "
          f"{'✓' if int(out) == expected else '✗'}")

print(f"\n→ Two layers solve XOR perfectly!")
print(f"  Each hidden neuron computes a linearly separable function.")
print(f"  The output neuron combines them — also linearly separable.")
print(f"  Minsky & Papert's barrier is broken by adding ONE layer.")

In [ ]:
# Visualize the two-layer solution geometrically
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x_range = np.linspace(-0.3, 1.3, 100)

# --- Hidden unit 1: OR ---
ax = axes[0]
for x, y in zip(X_xor, y_xor):
    h, _ = net.forward(x)
    color = 'blue' if h[0] == 1 else 'red'
    ax.scatter(*x, c=color, s=300, edgecolors='black', linewidth=2, zorder=5)

ax.plot(x_range, 0.5 - x_range, 'k-', linewidth=2)
ax.fill_between(x_range, 0.5 - x_range, 2, alpha=0.1, color='blue')
ax.fill_between(x_range, -1, 0.5 - x_range, alpha=0.1, color='red')
ax.set_title('Hidden neuron 1: OR\n$x_1 + x_2 > 0.5$')
ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

# --- Hidden unit 2: NAND ---
ax = axes[1]
for x, y in zip(X_xor, y_xor):
    h, _ = net.forward(x)
    color = 'blue' if h[1] == 1 else 'red'
    ax.scatter(*x, c=color, s=300, edgecolors='black', linewidth=2, zorder=5)

ax.plot(x_range, 1.5 - x_range, 'k-', linewidth=2)
ax.fill_between(x_range, -1, 1.5 - x_range, alpha=0.1, color='blue')
ax.fill_between(x_range, 1.5 - x_range, 2, alpha=0.1, color='red')
ax.set_title('Hidden neuron 2: NAND\n$-x_1 - x_2 > -1.5$')
ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

# --- Output: AND of hidden units (= XOR in hidden space) ---
ax = axes[2]
# Plot in hidden space (h1, h2)
for x, y in zip(X_xor, y_xor):
    h, out = net.forward(x)
    color = 'blue' if out == 1 else 'red'
    marker = 'o' if out == 1 else 's'
    ax.scatter(h[0], h[1], c=color, s=300, edgecolors='black', linewidth=2,
              marker=marker, zorder=5)
    ax.annotate(f'({int(x[0])},{int(x[1])})→{int(out)}', 
               xy=(h[0], h[1]), xytext=(10, 10),
               textcoords='offset points', fontsize=10)

ax.plot(x_range, 1.5 - x_range, 'k-', linewidth=2)
ax.set_title('Hidden space: NOW linearly separable!\nOutput: AND($h_1$, $h_2$)')
ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('$h_1$ (OR)'); ax.set_ylabel('$h_2$ (NAND)')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

plt.suptitle('How Two Layers Solve XOR: Transform to a Linearly Separable Space',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: The hidden layer TRANSFORMS the input space.")
print("In the original (x1, x2) space, XOR is not linearly separable.")
print("In the hidden (h1, h2) space, XOR becomes linearly separable!")
print("\nThis is the fundamental idea behind ALL deep learning:")
print("  → Learn a representation where the problem becomes easy.")

## Part 5: From Limitations to the Universal Approximation Theorem

Minsky & Papert showed: **1 layer → only linear functions.**

The Universal Approximation Theorem (Cybenko 1989) showed:
**2 layers (with nonlinear activation) → ANY continuous function.**

Let's demonstrate with a non-linear decision boundary that a 2-layer
network can learn but a single-layer perceptron cannot.

In [ ]:
def sigmoid(x):
    """Sigmoid activation function."""
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))


class TwoLayerNetwork:
    """A simple 2-layer network trained with gradient descent.
    
    Architecture: Input(2) → Hidden(n_hidden, sigmoid) → Output(1, sigmoid)
    Trained with binary cross-entropy loss.
    """
    
    def __init__(self, n_hidden: int = 10, lr: float = 1.0):
        self.W1 = np.random.randn(n_hidden, 2) * 0.5
        self.b1 = np.zeros(n_hidden)
        self.W2 = np.random.randn(n_hidden) * 0.5
        self.b2 = 0.0
        self.lr = lr
    
    def forward(self, x):
        self.h = sigmoid(self.W1 @ x + self.b1)
        self.out = sigmoid(np.dot(self.W2, self.h) + self.b2)
        return self.out
    
    def train_step(self, x, y):
        # Forward
        pred = self.forward(x)
        
        # Backward (gradient descent)
        d_out = pred - y  # dL/d_out for BCE
        d_W2 = d_out * self.h
        d_b2 = d_out
        d_h = d_out * self.W2 * self.h * (1 - self.h)
        d_W1 = np.outer(d_h, x)
        d_b1 = d_h
        
        # Update
        self.W2 -= self.lr * d_W2
        self.b2 -= self.lr * d_b2
        self.W1 -= self.lr * d_W1
        self.b1 -= self.lr * d_b1


# Generate XOR-like (circular) data — NOT linearly separable
np.random.seed(42)
n = 200
X_circle = np.random.randn(n, 2) * 2
y_circle = ((X_circle[:, 0]**2 + X_circle[:, 1]**2) < 3).astype(float)

# Train single-layer (linear classifier)
w_linear = np.zeros(2)
b_linear = 0.0
for epoch in range(200):
    for xi, yi in zip(X_circle, y_circle):
        target = 2 * yi - 1
        pred = 1 if np.dot(w_linear, xi) + b_linear >= 0 else -1
        if pred != target:
            w_linear += 0.01 * target * xi
            b_linear += 0.01 * target

# Train two-layer network
net2 = TwoLayerNetwork(n_hidden=20, lr=0.5)
for epoch in range(300):
    idx = np.random.permutation(n)
    for i in idx:
        net2.train_step(X_circle[i], y_circle[i])

# Visualize both
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, title, predict_fn in [
    (axes[0], 'Single-Layer Perceptron\n(can only learn linear boundary)', 
     lambda x: 1 if np.dot(w_linear, x) + b_linear >= 0 else 0),
    (axes[1], 'Two-Layer Network\n(learns non-linear boundary)',
     lambda x: 1 if net2.forward(x) >= 0.5 else 0),
]:
    # Decision boundary
    xx, yy = np.meshgrid(np.linspace(-4, 4, 150), np.linspace(-4, 4, 150))
    Z = np.array([predict_fn(np.array([x, y])) for x, y in zip(xx.ravel(), yy.ravel())])
    Z = Z.reshape(xx.shape)
    
    cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_light)
    ax.scatter(X_circle[y_circle==0, 0], X_circle[y_circle==0, 1], 
              c='red', s=20, alpha=0.6, label='Outside')
    ax.scatter(X_circle[y_circle==1, 0], X_circle[y_circle==1, 1], 
              c='blue', s=20, alpha=0.6, label='Inside')
    
    # True boundary
    theta = np.linspace(0, 2*np.pi, 100)
    ax.plot(np.sqrt(3)*np.cos(theta), np.sqrt(3)*np.sin(theta), 
            'k--', linewidth=2, label='True boundary')
    
    # Accuracy
    preds = np.array([predict_fn(x) for x in X_circle])
    acc = np.mean(preds == y_circle)
    ax.set_title(f'{title}\nAccuracy: {acc:.1%}')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.set_aspect('equal')

plt.suptitle('Minsky & Papert\'s Limitation Overcome: Non-Linear Decision Boundary',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("The single-layer perceptron can only draw a straight line — fails on circular data.")
print("The two-layer network learns a curved boundary — succeeds!")
print("\nThis is the essence of the Universal Approximation Theorem:")
print("  With enough hidden neurons and a nonlinear activation,")
print("  a 2-layer network can approximate ANY continuous function.")

## Summary: The Arc from Limitation to Revolution

| What Minsky & Papert Proved | What It Motivated |
|---|---|
| XOR impossible (single layer) | Multi-layer networks (Paper #6) |
| Parity requires full observation | Need for global feature extraction |
| Connectedness impossible (finite order) | Hierarchical feature learning (deep nets) |
| Group Invariance limits recognition | Weight sharing in CNNs (Papers #7, #10) |
| Single layer = linear functions only | Universal Approximation Theorem (1989) |

**The lesson**: Understanding limitations is the first step toward overcoming them.

**Next**: Paper #5 (Hopfield Networks) and Paper #6 (Backpropagation) will show
how the field recovered from the AI winter and solved these limitations.